# Baseline judging with Gemini (no GPU needed)

The baseline replies are already generated. This notebook only **grades** them with Gemini so the baseline can be compared fairly with Novelty 2.

**Any Google account works, and a CPU runtime is enough** (Runtime > Change runtime type > CPU). No T4 is needed.

**Once per account:**
1. In Google Drive create `MyDrive/refusalbench-results/` and upload the folder `qwen15_7b_baseline` into it (unzip `Benchmark-Baseline-Reproduction/qwen15_7b_baseline.zip` on your PC first). It must contain `target_outputs.jsonl`.
2. Colab left bar > key icon > **Add new secret**: name `GEMINI_API_KEY`, paste your key, switch "Notebook access" on.

**Every day:** Runtime > **Run all**. It stops by itself when the free daily limit (500 requests per key) is used up, and prints how many replies are graded. Run it again tomorrow, or put a key from another Google project in the secret and run again now.

**Stay on ONE account for the whole judging**, because the progress (`judged_outputs.jsonl`) lives in that account's Drive. To change account, upload the latest downloaded zip's folder to the new Drive first.

Done when it prints 100% graded. The last step downloads a zip: unzip it into `Benchmark-Baseline-Reproduction/` on your PC.

In [ ]:
# BASELINE JUDGING - judge-only, NO GPU needed (any Google account, Runtime type: CPU is fine).
# Grades Qwen's baseline replies with Gemini. Safe to re-run every day: finished work is kept and skipped.
import os, subprocess, sys

REPO = 'aidepartmentrah-pixel/refusalbench-reproduction'   # the repo that holds this code
OUT  = '/content/drive/MyDrive/refusalbench-results'        # the folder qwen15_7b_baseline/ must be inside this folder
RUN  = f'{OUT}/qwen15_7b_baseline'


def run(cmd, **kw):
    """Run a command and print its output LIVE."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'}, **kw)
    while True:
        chunk = os.read(p.stdout.fileno(), 4096)
        if not chunk:
            break
        print(chunk.decode('utf-8', 'replace'), end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {cmd}')


def count(path):
    return sum(1 for _ in open(path, encoding='utf-8')) if os.path.exists(path) else 0


def main():
    from google.colab import drive
    drive.mount('/content/drive')

    # 1) The saved baseline replies must be on THIS account's Drive
    if not os.path.exists(f'{RUN}/target_outputs.jsonl'):
        print('\n>>> NOT STARTED: the baseline replies are not on this Google account\'s Drive.\n'
              '    Do this once: in Drive create the folder  MyDrive/refusalbench-results  and upload the folder\n'
              '    "qwen15_7b_baseline" into it (unzip Benchmark-Baseline-Reproduction/qwen15_7b_baseline.zip on your PC first).\n'
              f'    It must contain: {RUN}/target_outputs.jsonl\n    Then run this cell again.')
        return
    print(f'Baseline replies found: {count(f"{RUN}/target_outputs.jsonl")} replies.')

    # 2) The Gemini key (Colab secret named GEMINI_API_KEY)
    try:
        from google.colab import userdata
        os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
        print('Gemini key found.')
    except Exception:
        print('\n>>> NOT STARTED: the Colab secret GEMINI_API_KEY was not found.\n'
              '    1. Click the KEY icon in the left bar of Colab.\n'
              '    2. "Add new secret": Name = GEMINI_API_KEY (exactly), Value = your key.\n'
              '    3. Switch ON the "Notebook access" toggle next to it.\n'
              '    4. Run this cell again.\n'
              '    (No key yet? Create a free one at aistudio.google.com > Get API key.)')
        return

    # 3) Code + packages (light: no model is loaded)
    token = ''
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN') or ''          # only needed if the repo is private
    except Exception:
        pass
    url = f'https://{token + "@" if token else ""}github.com/{REPO}.git'
    if not os.path.exists('/content/refusalbench-reproduction'):
        run(['git', 'clone', '-q', url, '/content/refusalbench-reproduction'])
    os.chdir('/content/refusalbench-reproduction')
    run(['git', 'pull', '-q'])
    run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'pyyaml', 'google-genai', 'python-dotenv'])

    # 4) Judge. --accept-partial = do NOT generate again: score the replies that exist (the 40 that failed earlier are
    #    excluded). It stops by itself when the free daily quota (500 requests per day per key) is used up.
    run([sys.executable, 'scripts/run_all.py', '--config',
         'Benchmark-Baseline-Reproduction/configs/qwen15_7b_baseline.yaml', '--accept-partial', '--out', OUT])

    # 5) How far are we? And a zip to keep on your PC (the Drive copy is already saved)
    n_t, n_j = count(f'{RUN}/target_outputs.jsonl'), count(f'{RUN}/judged_outputs.jsonl')
    print(f'\n=== PROGRESS: {n_j} of {n_t} baseline replies are graded ({n_j / max(n_t, 1):.0%}).')
    print('Done when this reaches 100%. If it stopped on the daily limit: run this cell again tomorrow, or replace the '
          'GEMINI_API_KEY secret with a key from ANOTHER Google project and run again now.')
    try:
        from google.colab import files
        run(f'cd {OUT} && zip -qr /content/qwen15_7b_baseline_judged.zip qwen15_7b_baseline', shell=True)
        files.download('/content/qwen15_7b_baseline_judged.zip')
    except Exception as e:
        print('download skipped:', e)


main()
